In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split


Handle Missing Value

In [ ]:
categorical_cols = df.select_dtypes(include = ["object"]).columns
numerical_cols = df.select_dtypes(include = ["float64"]).columns

In [ ]:
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy = "mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

In [ ]:
cat_imp = SimpleImputer(strategy = "most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

EDA - exploratory data analysis

In [ ]:
classes_count = df["Loan_Approved"].value_counts()

plt.pie(classes_count,labels= ["No","Yes"],autopct = "%1.1f%%")
plt.title("is loan approved or not")

In [ ]:
# analyse categories

gender_cnt = df["Gender"].value_counts()
ax = sns.barplot(gender_cnt)
ax.bar_label(ax.containers[0])

In [ ]:
# analyse income

sns.histplot(data = df, x = "Applicant_Income",bins = 20)

In [ ]:
sns.histplot(data = df,x = "Coapplicant_Income",bins = 20)

In [ ]:
# outliers -  box plot

sns.boxplot(data = df,x = "Loan_Approved",y = "Applicant_Income")

In [ ]:
fig, axes = plt.subplots(3,2)
sns.boxplot(ax = axes[0,0],data = df, x = "Loan_Approved",y = "Applicant_Income")
sns.boxplot(ax = axes[0,1],data = df, x = "Loan_Approved",y = "Credit_Score")
sns.boxplot(ax = axes[1,0],data = df, x = "Loan_Approved",y = "DTI_Ratio")
sns.boxplot(ax = axes[1,1],data = df, x = "Loan_Approved",y = "Savings")
sns.boxplot(ax = axes[2,0],data = df, x = "Loan_Approved",y = "Age")
sns.boxplot(ax = axes[2,1],data = df, x = "Loan_Approved",y = "Loan_Amount")
plt.tight_layout()

Credit score with loan approved

In [ ]:
sns.histplot(data = df,x = "Credit_Score",hue = "Loan_Approved",bins = 20,multiple = "dodge")

In [ ]:
df = df.drop("Applicant_ID",axis = 1)

In [ ]:
Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
le = LabelEncoder()
df["Education_Level"] = le.fit_transform(df["Education_Level"])
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])

In [ ]:
cols = ["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]

ohe = OneHotEncoder(drop = "first",sparse_output = False,handle_unknown = "ignore")
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(cols),index = df.index)
df = pd.concat([df.drop(columns = cols),encoded_df],axis = 1)

Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include = "number")
corr_matrix = num_cols.corr()

plt.figure(figsize = (15,8))
sns.heatmap(corr_matrix,annot = True,fmt = ".2f",cmap = "coolwarm")

Train-Test-Split + Feature Scaling

In [ ]:
x = df.drop("Loan_Approved",axis = 1)
y = df["Loan_Approved"]

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

Train and Evaluate Models

In [ ]:
# Logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score,precision_score,recall_score,f1_score
log_model = LogisticRegression()
log_model.fit(x_train_scaled,y_train)

y_pred = log_model.predict(x_test_scaled)

#Evalute
print("Logistic Regression Model")
print("Precision Score: ", precision_score(y_test,y_pred))
print("Recall Score: ", recall_score(y_test,y_pred))
print("F1 Score: ", f1_score(y_test,y_pred))
print("Accurracy Score: ", accuracy_score(y_test,y_pred))
print("CM: ", confusion_matrix(y_test,y_pred))

In [ ]:
#knn
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors = 5)
knn_model.fit(x_train_scaled,y_train)
y_pred = knn_model.predict(x_test_scaled)

print("Logistic Regression Model")
print("Precision Score: ", precision_score(y_test,y_pred))
print("Recall Score: ", recall_score(y_test,y_pred))
print("F1 Score: ", f1_score(y_test,y_pred))
print("Accurracy Score: ", accuracy_score(y_test,y_pred))
print("CM: ", confusion_matrix(y_test,y_pred))

In [ ]:
#naive bayes
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(x_train_scaled,y_train)
y_pred = nb_model.predict(x_test_scaled)

print("Logistic Regression Model")
print("Precision Score: ", precision_score(y_test,y_pred))
print("Recall Score: ", recall_score(y_test,y_pred))
print("F1 Score: ", f1_score(y_test,y_pred))
print("Accurracy Score: ", accuracy_score(y_test,y_pred))
print("CM: ", confusion_matrix(y_test,y_pred))

Best Model on the basis of Precision => Naive Bayes

Feature Engineering

In [ ]:
# add or transform features
df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2

x = df.drop(columns = ["Loan_Approved","Credit_Score","DTI_Ratio"])
y = df["Loan_Approved"]

# Train test split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

# scaling 
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score,precision_score,recall_score,f1_score
log_model = LogisticRegression()
log_model.fit(x_train_scaled,y_train)

y_pred = log_model.predict(x_test_scaled)

#Evalute
print("Logistic Regression Model")
print("Precision Score: ", precision_score(y_test,y_pred))
print("Recall Score: ", recall_score(y_test,y_pred))
print("F1 Score: ", f1_score(y_test,y_pred))
print("Accurracy Score: ", accuracy_score(y_test,y_pred))
print("CM: ", confusion_matrix(y_test,y_pred))

In [ ]:
#knn
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors = 5)
knn_model.fit(x_train_scaled,y_train)
y_pred = knn_model.predict(x_test_scaled)

print("Logistic Regression Model")
print("Precision Score: ", precision_score(y_test,y_pred))
print("Recall Score: ", recall_score(y_test,y_pred))
print("F1 Score: ", f1_score(y_test,y_pred))
print("Accurracy Score: ", accuracy_score(y_test,y_pred))
print("CM: ", confusion_matrix(y_test,y_pred))

In [ ]:
#naive bayes
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(x_train_scaled,y_train)
y_pred = nb_model.predict(x_test_scaled)

print("Logistic Regression Model")
print("Precision Score: ", precision_score(y_test,y_pred))
print("Recall Score: ", recall_score(y_test,y_pred))
print("F1 Score: ", f1_score(y_test,y_pred))
print("Accurracy Score: ", accuracy_score(y_test,y_pred))
print("CM: ", confusion_matrix(y_test,y_pred))